# 최근 3개월 예측과 포아송 회귀 비교

**의정이슈레이더 예측 모델 검증 노트북**

이 노트북은 다음 두 실험을 분리해서 설명하고, 같은 평가셋에서 다시 비교할 수 있는 코드를 제공합니다.

1. **단일 이슈 프로토타입**: 최근 3개월 가중평균(`weighted_mean_3m`)
2. **대표 10개 이슈 통합 모델**: 포아송 회귀(`poisson_regression`)

> 핵심 주의사항: 기존 기록의 MAE 4.256과 MAE 7.1224는 평가 대상이 각각 단일 이슈와 대표 10개 이슈로 다르므로, 숫자만 보고 직접 우열을 판단하면 안 됩니다.

## 1. 비교 원칙

- 예측 시점 이후 데이터는 학습에 사용하지 않습니다.
- 각 Fold에서 **과거 월로 학습하고 바로 다음 월을 예측**합니다.
- 동일한 `issue_id × month` 행에 대해 모델별 MAE와 RMSE를 계산합니다.
- 모델 선택의 1차 기준은 해석하기 쉬운 **MAE**, 큰 오차 점검에는 **RMSE**를 사용합니다.

| 모델 | 의미 | 장점 | 한계 |
|---|---|---|---|
| 전월 유지 | 직전 월 건수를 그대로 예측 | 가장 단순한 기준선 | 추세 변화 반영이 약함 |
| 3개월 평균 | 최근 3개월 평균 | 변동 완화 | 최근 월의 중요도가 동일함 |
| 3개월 가중평균 | 최근 월에 더 큰 가중치 | 최근 추세 반영 | 이슈 간 차이를 학습하지 않음 |
| 포아송 회귀 | 여러 시계열 특징으로 0 이상 건수 예측 | 다중 이슈·카운트 자료에 적합 | 특징과 정규화 설정이 필요함 |

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MEMBERS_DIR = PROJECT_ROOT / "data/forecast"                        # 노트북 07이 만든 이슈 구성 청크 캐시 (5절 입력)
OFFICIAL_METRICS = PROJECT_ROOT / "model/forecast/forecast_metrics.json"  # 원본 32번 공식 평가 결과 (7절)
# 선택 입력: 원본 파이프라인 산출물을 data/forecast/에 두면 그 파일을 우선 사용
MULTI_PANEL = PROJECT_ROOT / "data/forecast/multi_issue_panel.csv"            # 원본 31번 패널
SINGLE_BACKTEST = PROJECT_ROOT / "data/forecast/birth_rate_forecast_backtest.csv"  # 원본 23번 단일 이슈 결과

for path in [MEMBERS_DIR, OFFICIAL_METRICS, MULTI_PANEL, SINGLE_BACKTEST]:
    print(f"{'존재' if path.exists() else '없음':4s} | {path.relative_to(PROJECT_ROOT)}")

## 2. 공통 평가 함수

MAE와 RMSE는 모두 언급 **건수 단위**입니다.

- $MAE = \frac{1}{n}\sum |y_i-\hat{y}_i|$
- $RMSE = \sqrt{\frac{1}{n}\sum(y_i-\hat{y}_i)^2}$

In [ ]:
def count_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "평가행": int(len(y_true)),
    }


def first_existing(columns, candidates, label):
    for name in candidates:
        if name in columns:
            return name
    raise KeyError(f"{label} 열을 찾지 못했습니다. 현재 열: {list(columns)}")


def add_history_features(panel, issue_col, month_col, target_col):
    """현재 월 정답을 보지 않고, 과거 월 언급량으로만 특징을 생성합니다."""
    work = panel.copy()
    work[month_col] = pd.to_datetime(work[month_col])
    work[target_col] = pd.to_numeric(work[target_col], errors="coerce").fillna(0).clip(lower=0)
    work = work.sort_values([issue_col, month_col]).reset_index(drop=True)
    grouped = work.groupby(issue_col, sort=False)[target_col]

    work["lag1"] = grouped.shift(1)
    work["mean_3m"] = grouped.transform(lambda s: s.shift(1).rolling(3, min_periods=3).mean())
    work["weighted_mean_3m"] = grouped.transform(
        lambda s: s.shift(1).rolling(3, min_periods=3).apply(
            lambda x: np.average(x, weights=[1, 2, 3]), raw=True
        )
    )
    work["std_3m"] = grouped.transform(lambda s: s.shift(1).rolling(3, min_periods=3).std(ddof=0))
    work["active_ratio_3m"] = grouped.transform(
        lambda s: s.shift(1).rolling(3, min_periods=3).apply(lambda x: np.mean(x > 0), raw=True)
    )
    work["trend_1m"] = grouped.shift(1) - grouped.shift(2)
    work["month_number"] = work[month_col].dt.month
    return work

## 3. 기존 프로젝트 결과 확인

아래 표는 프로젝트에서 이미 확정해 기록한 값입니다. 두 행은 **평가 범위가 다르므로 직접 비교용 표가 아니라 실험 이력 표**입니다.

In [ ]:
recorded_results = pd.DataFrame([
    {
        "평가 범위": "출산율 단일 이슈",
        "모델": "최근 3개월 가중평균",
        "MAE": 4.256,
        "RMSE": 6.084,
        "검증 방식": "단일 시계열 Walk-forward",
        "직접 비교 가능": "아니오",
    },
    {
        "평가 범위": "대표 10개 이슈",
        "모델": "포아송 회귀",
        "MAE": 7.1224,
        "RMSE": 10.0425,
        "검증 방식": "시간순 Rolling Backtest 10 folds",
        "직접 비교 가능": "아니오",
    },
])
display(recorded_results)
print("주의: MAE 4.256 < 7.1224만으로 3개월 가중평균이 더 좋다고 결론 내릴 수 없습니다.")

## 4. 단일 이슈의 최근 3개월 가중평균 검증

원본 `src/23_forecast_issue.py`의 단일 이슈 Backtest 결과 파일(`data/forecast/birth_rate_forecast_backtest.csv`)이 있으면 불러와 다시 계산합니다. 이 파일은 제출물에 포함하지 않았으므로 기본 실행에서는 3절의 기록값(팀 실험 기록)을 그대로 씁니다.

In [ ]:
if SINGLE_BACKTEST.exists():
        single = pd.read_csv(SINGLE_BACKTEST)
        print("열:", single.columns.tolist())
        display(single.head())

        actual_candidates = ["actual", "y_true", "actual_count", "mention_count"]
        pred_candidates = ["weighted_mean_3m", "prediction", "y_pred", "predicted_count"]
        try:
            actual_col = first_existing(single.columns, actual_candidates, "실제값")
            pred_col = first_existing(single.columns, pred_candidates, "예측값")
            valid = single[[actual_col, pred_col]].dropna()
            print("재계산:", count_metrics(valid[actual_col], valid[pred_col]))
        except KeyError as error:
            print("자동 재계산 생략:", error)
else:
    print("단일 이슈 Backtest 파일이 없어 기록값을 사용합니다: MAE=4.256, RMSE=6.084")

## 5. 동일 평가셋에서 네 모델 공정 비교

이 부분이 실제로 **3개월 예측과 포아송 회귀를 비교하는 핵심 코드**입니다.

- 입력: 대표 10개 이슈의 월별 언급량 패널. 기본은 노트북 07(8-1)이 `data/forecast/members_*.json`에 저장한 이슈 구성 청크로 만듭니다(DB 불필요). 원본 31번 패널 `data/forecast/multi_issue_panel.csv`를 두면 그 파일을 씁니다.
- 평가: 마지막 최대 10개 월을 Fold로 사용
- 학습: 각 예측 월보다 과거인 행만 사용
- 모델: 전월 유지, 3개월 평균, 3개월 가중평균, 포아송 회귀

> 이 절의 포아송 회귀는 비교용 간이 구성(과거 언급량 특징 7개 + 이슈 원-핫, `alpha=1.0`)입니다. 서비스(노트북 07 · 8-4)의 포아송 회귀는 원본 32번과 같은 특징 19개 · `log1p` 변환 · `alpha=0.5` 구성이라 설정이 다릅니다.


In [ ]:
def evaluate_same_panel(panel, max_folds=10):
    issue_col = first_existing(panel.columns, ["issue_id", "issue_key", "issue"], "이슈 ID")
    month_col = first_existing(panel.columns, ["feature_month", "month", "year_month", "date", "ds"], "월")
    target_col = first_existing(
        panel.columns,
        ["mention_count", "monthly_count", "evidence_count", "count", "target_count", "y"],
        "언급량",
    )

    work = add_history_features(panel, issue_col, month_col, target_col)
    work = work.dropna(subset=["lag1", "mean_3m", "weighted_mean_3m", "std_3m", "active_ratio_3m"])
    months = sorted(work[month_col].unique())
    test_months = months[-min(max_folds, len(months)):]
    rows = []

    numeric_features = ["lag1", "mean_3m", "weighted_mean_3m", "std_3m", "active_ratio_3m", "trend_1m", "month_number"]
    categorical_features = [issue_col]

    for fold, test_month in enumerate(test_months, start=1):
        train = work[work[month_col] < test_month].dropna(subset=numeric_features + [target_col])
        test = work[work[month_col] == test_month].dropna(subset=numeric_features + [target_col])
        if train.empty or test.empty:
            continue

        preprocessor = ColumnTransformer([
            ("number", StandardScaler(), numeric_features),
            ("issue", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ])
        poisson = Pipeline([
            ("preprocess", preprocessor),
            ("model", PoissonRegressor(alpha=1.0, max_iter=2000)),
        ])
        poisson.fit(train[numeric_features + categorical_features], train[target_col])

        predictions = {
            "전월 유지": test["lag1"].to_numpy(),
            "3개월 평균": test["mean_3m"].to_numpy(),
            "3개월 가중평균": test["weighted_mean_3m"].to_numpy(),
            "포아송 회귀": poisson.predict(test[numeric_features + categorical_features]),
        }
        for model_name, pred in predictions.items():
            for idx, actual, predicted in zip(test.index, test[target_col], pred):
                rows.append({
                    "fold": fold,
                    "test_month": pd.Timestamp(test_month),
                    "issue_id": test.loc[idx, issue_col],
                    "model": model_name,
                    "actual": float(actual),
                    "predicted": float(max(0, predicted)),
                })

    predictions = pd.DataFrame(rows)
    if predictions.empty:
        raise ValueError("평가 가능한 행이 없습니다. 최소 4개월 이상의 이슈별 월별 데이터가 필요합니다.")

    metrics = []
    for model_name, group in predictions.groupby("model"):
        result = count_metrics(group["actual"], group["predicted"])
        result["모델"] = model_name
        metrics.append(result)
    metrics = pd.DataFrame(metrics).set_index("모델").sort_values("MAE")
    return metrics, predictions

In [ ]:
import hashlib

# 노트북 07(8-1)이 저장한 대표 이슈 구성 청크(member) 캐시 → 이슈별 월별 언급량 패널
# 파일 이름 = md5("테이블|이슈 검색어") (노트북 07 issue_members와 같은 규칙)
PG_TABLE = "council_arctic_raw_v1"
CUTOFF_MONTH = "2025-12"                      # 회의록 기준일이 속한 달 (노트북 07과 동일)
ISSUE_QUERIES = {
    "birth_rate": "출산율 저하 대응 정책",
    "population_decline": "인구감소 지역소멸 대응 정책",
    "youth_jobs": "청년 일자리 지역정착 지원 정책",
    "jeonse_fraud": "전세사기 피해자 지원 대책",
    "regional_healthcare": "지역의료 공백 필수의료 대책",
    "care_policy": "돌봄 공백 통합돌봄 지원 정책",
    "school_violence": "학교폭력 예방 피해자 보호 대책",
    "climate_neutrality": "기후위기 탄소중립 대응 정책",
    "flood_safety": "집중호우 침수 피해 예방 대책",
    "small_business": "소상공인 경영 안정 지원 정책",
}


def panel_from_members() -> pd.DataFrame:
    rows = []
    for issue_id, query in ISSUE_QUERIES.items():
        cache = MEMBERS_DIR / f"members_{hashlib.md5(f'{PG_TABLE}|{query}'.encode()).hexdigest()}.json"
        members = json.loads(cache.read_text(encoding="utf-8"))
        months = pd.Series([m["meeting_date"][:7] for m in members if m.get("meeting_date") and m.get("council")])
        counts = months[months <= CUTOFF_MONTH].value_counts()
        # 첫 관측 달부터 기준 달까지 빠짐없이 (언급 없는 달은 0)
        for month in pd.period_range(counts.index.min(), CUTOFF_MONTH, freq="M").astype(str):
            rows.append({"issue_id": issue_id, "feature_month": month, "mention_count": int(counts.get(month, 0))})
    return pd.DataFrame(rows)

In [ ]:
same_scope_metrics = None
same_scope_predictions = None

if MULTI_PANEL.exists():
    panel = pd.read_csv(MULTI_PANEL)
    print(f"원본 31번 패널 사용: {len(panel):,}행")
else:
    panel = panel_from_members()
    print(f"노트북 07 이슈 구성 청크로 만든 패널 사용: {len(panel):,}행 "
          f"({panel['issue_id'].nunique()}개 이슈, {panel['feature_month'].min()} ~ {panel['feature_month'].max()})")
same_scope_metrics, same_scope_predictions = evaluate_same_panel(panel, max_folds=10)
months = same_scope_predictions["test_month"].dt.strftime("%Y-%m")
print(f"Fold {months.nunique()}개 ({months.min()} ~ {months.max()}), 모델별 평가 {len(same_scope_predictions) // 4}건")
display(same_scope_metrics.round(4))
print(f"동일 평가셋 MAE 기준 1위: {same_scope_metrics.index[0]}")

## 6. 비교 결과 시각화

실제 패널이 있을 때만 같은 평가셋의 MAE·RMSE 그래프를 생성합니다.

In [ ]:
if same_scope_metrics is not None:
    ax = same_scope_metrics[["MAE", "RMSE"]].plot(
        kind="bar", figsize=(10, 5), rot=0, color=["#6f7754", "#d95d52"]
    )
    ax.set_title("동일 다중 이슈 패널에서의 예측 모델 비교")
    ax.set_ylabel("오차(언급 건수)")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("시각화 생략: 다중 이슈 패널 필요")

## 7. 공식 다중 이슈 결과와 기준선 개선율

원본 파이프라인(`src/32`, 원본 31번 패널 180행)의 공식 평가 결과(`model/forecast/forecast_metrics.json`)입니다.

- MAE: **7.1224건**
- RMSE: **10.0425건**
- 전월 유지 기준선 대비 MAE: **29.83% 개선**
- 전월 유지 기준선 대비 RMSE: **24.31% 개선**

아래 코드는 개선율을 역산해 당시 전월 유지 기준선의 오차 규모를 확인합니다.

In [ ]:
official = json.loads(OFFICIAL_METRICS.read_text(encoding="utf-8"))["required_evaluation"]
poisson_mae = official["mention_count_mae"]
poisson_rmse = official["mention_count_rmse"]
mae_improvement = official["last_month_persistence_comparison"]["count_mae_improvement_percent"] / 100
rmse_improvement = official["last_month_persistence_comparison"]["count_rmse_improvement_percent"] / 100

previous_month_mae = poisson_mae / (1 - mae_improvement)
previous_month_rmse = poisson_rmse / (1 - rmse_improvement)

official_comparison = pd.DataFrame([
    {"모델": "전월 유지 기준선", "MAE": previous_month_mae, "RMSE": previous_month_rmse},
    {"모델": "포아송 회귀", "MAE": poisson_mae, "RMSE": poisson_rmse},
]).set_index("모델")
display(official_comparison.round(4))

## 8. 결론

1. **단일 출산율 이슈**에서는 최근 추세만 반영하는 3개월 가중평균이 MAE 4.256으로 선택되었습니다(팀 실험 기록).
2. **대표 10개 이슈 통합 실험**(원본 파이프라인)에서는 이슈별 과거 활동 특징을 함께 학습한 포아송 회귀가 MAE 7.1224를 기록했습니다.
3. 두 값은 평가 범위가 달라 직접 비교하면 안 됩니다.
4. 그래서 5절에서 **동일 패널·동일 Fold**(대표 10개 이슈, 2025-03 ~ 2025-12의 10 Fold, 모델별 100건)로 다시 비교했습니다.
   기본 실행 결과 포아송 회귀가 MAE 4.79 · RMSE 6.23으로 3개월 평균(5.18 · 7.33), 3개월 가중평균(5.20 · 7.46), 전월 유지(6.21 · 9.29)보다 오차가 작았습니다.
5. 최종 서비스는 여러 이슈에 공통 적용해야 하므로 다중 이슈 시간순 평가에서 선택된 포아송 회귀를 사용합니다.
   단, 5절의 포아송 회귀는 비교용 간이 구성이고 서비스(노트북 07)의 포아송 회귀와 특징·설정이 다릅니다.

### 발표용 한 문장

> 단일 이슈에서는 최근 3개월 가중평균이 안정적이었지만, 최종 서비스는 여러 이슈를 동시에 예측해야 하므로 동일한 시간순 검증에서 다중 특징을 학습하고 전월 유지 기준선보다 MAE를 29.83% 낮춘 포아송 회귀를 채택했습니다.


## 9. 원본 프로젝트 코드 위치

- `src/23_forecast_issue.py`: 단일 이슈 기준선 비교와 다음 달 예측
- `src/31_build_multi_issue_panel.py`: 대표 이슈 월별 패널 생성
- `src/32_train_multi_issue_forecast.py`: 다중 이슈 Rolling Backtest와 포아송 회귀
- `notebooks/07_council_issue_radar.ipynb` 8절: 서비스용 이슈 구성 청크 · 예측 특징 · 포아송 학습과 다음 달 예측

제출용 노트북은 계산 과정을 설명하기 위한 것이며, 최종 수치 재현의 기준 구현은 위 Python 소스입니다.